# Workshop: iterators and generators

### Monday, February 26, 2024

In today's workshop we'll start talking about functional programming, starting with the basics of iterators and generators and talking about map and filter operations. Then, on Wednesday, we'll talk about the more complicated (and more powerful!) aspects of functional programming, including accumulators and lambda expressions.

## Problem 1: iter and next

Let's begin by reviewing the `__next__()` and `__iter__()` methods, which are central to iteration in Python.

Recall that an <b>iterator</b> is any object in Python that supports the `__next__()` method.

Create a class `Powers` that iterates over powers of an integer.
This class should have:

- An `__init__()` method that takes two integers as its arguments: a base `b` (any integer) and `maxiters` (non-negative integer), and sets `b` as an instance attribute `base`.
- A `__next__()` method, that returns `base` raised to a power. The first time `__next__` is called should return `base**0`, then `base**1` then `base**2`, etc. Once `maxiters` powers have been returned, the `Powers` object should stop returning values (i.e., raise a `StopIteration` error to signal that there are no more values to return). Read the documentation here for details: https://docs.python.org/3/library/stdtypes.html#iterator.__next__

In [7]:
class Powers:
    
    def __init__(self, b, maxiters ):

        if not isinstance(b, (int, )):
            raise TypeError(f'b = {b} must be of type int.')
        
        if not isinstance(maxiters, (int, )):
            raise TypeError(f'maxiters = {maxiters} must be of type int.')
        if maxiters < 0:
            raise ValueError(f'maxiters = {maxiters} must be non-negative.')

        self.b = b
        self.next_value = 1
        self.calls_so_far = 0
        self.maxiters = maxiters
        
    def __next__(self):
        
        while (self.calls_so_far <= self.maxiters):
            self.next_value = self.b ** self.calls_so_far
            self.calls_so_far += 1
            return self.next_value
        else:
            raise StopIteration
            

Now, consider the following code. What should it do?

Once you've made your prediction, try running it.

In [8]:
p = Powers(2, 10)

[next(p) for i in range(10)]

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

The above code is possible because `Powers` supports the `__next__` method. It is an <i>iterator</i>.

But it is not an <i>iterable</i>. That is, we cannot yet write something like `p = Powers(2); for x in p:...`

In [ ]:
p = Powers(3, 10)
for x in p: # This should raise an error because Powers doesn't have an __iter__ method.
    print(x)

Recall from lecture that for us to be able to run the above code, `Powers` must support the `__iter__` method. Update your `Powers` class above so that `Powers` is an iterable, then try running the previous block of code again.

## Problem 2: list comprehensions and generator expressions

One distinction that was drawn in lecture was that between list comprehensions and generator expressions.
At first glance, these are very similar. Compare `[x for x in mylist]` with `(x for x in mylist)`.
In this exercise, we'll explore the difference between these.

Let's begin by running the following two blocks of code.

In [ ]:
# Map the function f(x)=x^2 onto the sequence [0,1,2,...,9]
[x**2 for x in range(10)] # Sidenote: this is an example of map: apply the square function to every element!

In [ ]:
(x**2 for x in range(10))

Okay, the first difference we see is that the list comprehension really does return a list (Jupyter notebook displays it and everything), while the generator expression returns a <i>generator object</i>. Let's look a bit more closely at that.

In [ ]:
gen = (x**2 for x in range(10))
for e in gen:
    print(e)

So gen is an iterable-- we can iterate over its elements.

And we know that we can iterate over the elements of a list.

What is the difference, then, between list comprehensions and generator expressions (other than the fact that lists and generators are different types...)?

The key difference becomes clear when we try to iterate over an infinite set.

Here's a slight variation on our `Powers` class above.

In [ ]:
class Evens: # Iterates over the even integers, starting with 0.
    def __init__( self ):
        self.n=0 # Counter for keeping track of how many evens we have given
    def __next__( self ):
        e = 2*self.n # The next even integer.
        self.n+=1 # Update our counter
        return e
    def __iter__( self ):
        return self # So that we can write for x in evens

Using the `Evens` class above, write a generator expression that generates the multiples of 4. Store the generator expression in a variable `g`.

In [ ]:
# TODO: code goes here.

Now, again using the `Evens` class above, write a <i>list comprehension</i> that generates the multiples of 4. Store it in a variable `t`. What happens? Why?

In [ ]:
#TODO: code goes here.

## Problem 3: the Catalan numbers

The Catalan numbers (https://en.wikipedia.org/wiki/Catalan_number) are a sequence given by
$$
C_n = \frac{ (2n)! }{ n!(n+1)! }
$$
for $n=0,1,2,\dots$.

Write a generator expression that enumerates all and only the odd Catalan numbers from among the first 100 Catalan numbers.

<b>Hint:</b> this is easiest done in two steps: a map operation followed by a filter operation.

<b>Second hint:</b> you may find it useful to use the following generator, which enumerates the numbers $0,1,2,\dots$

In [1]:
def natural_numbers(): # This is a generator. More on that in the next exercise.
    n=0
    while True:
        yield n
        n += 1

In [2]:
for i in natural_numbers():
    print(i)
    if i > 10:
        break

0
1
2
3
4
5
6
7
8
9
10
11


In [5]:
def catalan(n):
    if n == 0:
        return 1
    else:
        return int((4*n-2)*catalan(n-1)/(n+1))

In [6]:
catalan_numbers = [catalan(n) for n in range(10)]
print(catalan_numbers)


[1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]


## Problem 4: generators

In addition to generator expressions, we can create more interesting/powerful generators using the `yield` keyword. This allows us to write something that looks a lot like functions, but which stores internal state.

Generators are kind of between functions and objects.
A function, once it returns a value, "forgets" all the work it did-- any intermediate computations we did when producing our result disappear (unless we do something clever like store them in a file or in a global variable).
In contrast, a generator stores internal state, which remains accessible between return values.
This distinction is made by using the `yield` keyword instead of the `return` keyword.

Implement a generator called `power_gen` with the same behavior as our `Powers` object in Problem 1 above, except it should only take a single argument, the base `b` (i.e., it should enumerate an infinite set of powers of `b`).

In [7]:
def power_gen( b ):
    n=0
    while True:
        yield b**n
        n += 1

In [8]:
pg = power_gen( 2 )
next(pg) # next_power should be 0

1

In [9]:
next(pg) # next_power should be 1

2

In [10]:
next(pg) # next_power should be 2

4

In [11]:
# Internal states of generators do not "interfere" with one another.
pg2 = power_gen( 2 )

In [13]:
next(pg2)

2

In [14]:
next(pg)

8

In [15]:
next(pg2) 

4

In [16]:
next(pg)

16

<b>Discussion:</b> Comparing this with the `Powers` class from Problem 1, which seems like the more natural solution to the problem of enumerating powers of a number?